In [2]:
%cd ../
%load_ext autoreload
%autoreload 2

/home/christian/bachelor-project


In [3]:
# BiLSTM training for 12-hour wind vector forecasting

# This notebook trains `BiLSTMModel` on 10-min measurements to forecast the next 12 hours (72 steps)
# of u/v vector components per station.

import os
from omegaconf import OmegaConf
from loguru import logger
import pandas as pd

from src.database.database_service import DatabaseService
from src.weather_stations.weather_station_service import WeatherStationService
from src.measurements.measurement_service import MeasurementService
from src.model.variant.bilstm_model import BiLSTMModel


/home/christian/bachelor-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Load config and initialize services
cfg = OmegaConf.load("conf/config.yaml")

db = DatabaseService(cfg)
ws_service = WeatherStationService(cfg, db)
weather_stations = ws_service.load_from_database(only_relevant=True)

ms_service = MeasurementService(cfg, db, weather_stations)

logger.info(f"Loaded {len(weather_stations)} weather stations")


2025-09-12 17:55:05.344 | INFO     | src.weather_stations.weather_station_data_provider:load_from_database:228 - Loaded 50 weather stations from database
2025-09-12 17:55:05.357 | INFO     | __main__:<module>:10 - Loaded 50 weather stations


In [7]:
# Load measurements from DB
measurements_df = ms_service.load_all_measurements_from_database()
logger.info(f"Measurements: {len(measurements_df)} rows")


2025-09-12 17:55:49.949 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 1000000)
2025-09-12 17:56:07.166 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 2000000)
2025-09-12 17:56:24.626 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 3000000)
2025-09-12 17:56:41.268 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 4000000)
2025-09-12 17:56:57.818 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 5000000)
2025-09-12 17:57:14.863 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loa

,air_pressure,air_temperature_2m,air_temperature_5cm,average_wind_direction,average_wind_speed,dew_point_temperature,id,precipitation_duration,precipitation_indicator,quality_level,record_date,relative_humidity,station_id,sum_precipitation_height
0,1002.8,8.7,8.6,70.0,1.8,8.1,250001,10.0,1,3,2024-10-02 02:40:00,95.7,96,0.11
1,1002.7,8.7,8.6,60.0,2.0,8.0,250002,10.0,1,3,2024-10-02 02:50:00,95.6,96,0.06
2,1002.7,8.7,8.6,60.0,1.9,8.1,250003,10.0,1,3,2024-10-02 03:00:00,96.1,96,0.06
3,1002.8,8.7,8.6,70.0,1.9,8.1,250004,10.0,1,3,2024-10-02 03:10:00,96.1,96,0.12
4,1002.8,8.7,8.6,60.0,1.9,8.1,250005,10.0,1,3,2024-10-02 03:20:00,96.0,96,0.02


In [8]:
# Optional: downsample or filter timeframe for quicker training
start_date = pd.Timestamp('2023-01-01')
end_date = pd.Timestamp('2025-04-01')
train_df = measurements_df[(measurements_df['record_date']>=start_date) & (measurements_df['record_date']<=end_date)].copy()

test_df = measurements_df[measurements_df['record_date']>end_date].copy()

train_df.head()

,air_pressure,air_temperature_2m,air_temperature_5cm,average_wind_direction,average_wind_speed,dew_point_temperature,id,precipitation_duration,precipitation_indicator,quality_level,record_date,relative_humidity,station_id,sum_precipitation_height
0,1002.8,8.7,8.6,70.0,1.8,8.1,250001,10.0,1,3,2024-10-02 02:40:00,95.7,96,0.11
1,1002.7,8.7,8.6,60.0,2.0,8.0,250002,10.0,1,3,2024-10-02 02:50:00,95.6,96,0.06
2,1002.7,8.7,8.6,60.0,1.9,8.1,250003,10.0,1,3,2024-10-02 03:00:00,96.1,96,0.06
3,1002.8,8.7,8.6,70.0,1.9,8.1,250004,10.0,1,3,2024-10-02 03:10:00,96.1,96,0.12
4,1002.8,8.7,8.6,60.0,1.9,8.1,250005,10.0,1,3,2024-10-02 03:20:00,96.0,96,0.02


In [ ]:
# Instantiate BiLSTM model
model = BiLSTMModel(
    history_steps=72,   # 12 hours of history
    horizon_steps=72,   # 12 hours forecast
    station_embedding_dim=8,
    hidden_size=128,
    num_layers=2,
    dropout=0.1,
    batch_size=64,
    learning_rate=1e-3,
    num_epochs=5, 
    val_split=0.2, 
    early_stopping_patience=5, 
    early_stopping_min_delta=5e-5, 
    restore_best_weights=True  # increase for real training
)
model


In [46]:
# Train model
model.train(train_df)


2025-09-12 16:27:50.207 | INFO     | src.model.variant.bilstm_model:train:15 - Preparing training: train_sequences=3970761, val_sequences=991046, stations=48, feature_dim=10, device=cuda
2025-09-12 16:27:50.220 | INFO     | src.model.variant.bilstm_model:train:18 - Model initialized: hidden_size=128, num_layers=2, dropout=0.1, params=650,000
2025-09-12 16:27:50.220 | INFO     | src.model.variant.bilstm_model:train:26 - DataLoader ready: train_samples=3970761, val_samples=991046, batch_size=64, batches_per_epoch=62044
2025-09-12 16:27:50.220 | INFO     | src.model.variant.bilstm_model:train:30 - CUDA available: using GPU 'NVIDIA GeForce RTX 4080 SUPER'
2025-09-12 16:27:50.221 | INFO     | src.model.variant.bilstm_model:train:40 - Epoch 1/5 starting (lr=1.000e-03)
2025-09-12 16:28:06.359 | INFO     | src.model.variant.bilstm_model:train:55 - Epoch 1 [6204/62044] batch_loss=0.000171
2025-09-12 16:28:22.359 | INFO     | src.model.variant.bilstm_model:train:55 - Epoch 1 [12408/62044] batch_

In [47]:
# Save model
save_dir = "models/new_model"
os.makedirs(save_dir, exist_ok=True)
model.save(save_dir)
logger.info(f"Saved BiLSTM model to {save_dir}")


2025-09-12 16:43:06.659 | INFO     | __main__:<module>:5 - Saved BiLSTM model to models/new_model


In [10]:
new_model = BiLSTMModel(
    device=None   # increase for real training
)

new_model.load('models/first_good')

In [18]:
# Quick prediction demo for a subset (uses last 72 steps per station)
preds = new_model.predict(measurements_df)
preds.head()


/home/christian/bachelor-project/src/model/variant/bilstm_model.py:428: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("station_id", sort=False, group_keys=False).apply(


,station_id,record_date,u_pred,v_pred
0,96,2025-09-10 08:30:00,-0.483345,-0.562206
1,96,2025-09-10 08:40:00,-0.478859,-0.562057
2,96,2025-09-10 08:50:00,-0.481697,-0.563381
3,96,2025-09-10 09:00:00,-0.486183,-0.564758
4,96,2025-09-10 09:10:00,-0.493049,-0.566627


In [48]:
evaluation = model.evaluate(test_df)


2025-09-12 16:43:11.433 | INFO     | src.model.variant.bilstm_model:evaluate:14 - Evaluation: samples=1740060, batch_size=64, batches=27189
2025-09-12 16:43:58.909 | INFO     | src.model.variant.bilstm_model:evaluate:63 - Evaluation metrics: mae_u=0.0514, rmse_u=0.2704, mae_v=0.0783, rmse_v=0.2792, mae_speed=0.0571, rmse_speed=0.1463, mae_direction_deg=4.16°


In [11]:
per_h = new_model.evaluate_per_horizon(test_df, save_dir="artifacts/metrics2")

2025-09-12 18:01:09.423 | INFO     | src.model.variant.bilstm_model:evaluate_per_horizon:623 - Computed per-horizon metrics for 1..%d steps
2025-09-12 18:01:09.552 | INFO     | src.model.variant.bilstm_model:evaluate_per_horizon:662 - Saved per-horizon plots to /home/christian/bachelor-project/artifacts/metrics2


In [21]:
per_h = model.evaluate_per_horizon(test_df)

2025-09-12 15:19:12.368 | INFO     | src.model.variant.bilstm_model:evaluate_per_horizon:71 - Computed per-horizon metrics for 1..%d steps
